# AgentGuard: Day 1 setup and tool-calling test

**Goal for today:** get a local LLM running on Colab and check that it can call tools reliably. Everything else in AgentGuard depends on this.

By the end of this notebook you will have:
1. Ollama running on a Colab T4 GPU
2. `qwen2.5:7b` (agent model) and `nomic-embed-text` (embedding model) pulled
3. A 10-prompt tool-calling score and the average time per model call
4. A first mini agent loop (model calls a tool, gets the result, answers)
5. Results saved to Google Drive

This notebook runs on **Google Colab** or **Kaggle**; it detects which one automatically.

**Before you start:**
- **Colab:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`.
- **Kaggle:** in the right panel, `Session options → Accelerator → GPU T4 x2` and `Internet → On` (needs a phone-verified account), then `Run All`.

**Pass mark:** 7 or more out of 10 on the tool-calling test. If the model scores lower, change `MODEL` in Step 1 and rerun from Step 4.

## Step 1: Settings
Change `MODEL` here if you need to try a backup model (`llama3.1:8b` or `qwen3:8b`).

In [3]:
MODEL = "qwen2.5:7b"            # agent model (backup: "llama3.1:8b" or "qwen3:8b")
EMBED_MODEL = "nomic-embed-text"
DRIVE_DIR = "/content/drive/MyDrive/agentguard"   # used on Colab only

## Step 2: Check the GPU and set up storage
You should see a **Tesla T4** (or P100 on Kaggle) in the output.

- **On Colab:** results go to Google Drive, so they survive a disconnect.
- **On Kaggle:** results go to `/kaggle/working` (download them from the Output panel, or click *Save Version* to keep them).

In [4]:
import subprocess, os, shutil
if shutil.which("nvidia-smi") is None:
    raise SystemExit("NO GPU: this runtime is CPU-only. Go to Runtime > Change runtime type > "
                     "T4 GPU > Save (Kaggle: Settings > Accelerator > GPU T4 x2), then run all cells again. "
                     "If no GPU is available on Colab, use Kaggle.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip())

ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    WORK_DIR = "/kaggle/working"
    DRIVE_DIR = "/kaggle/working/agentguard"
else:
    WORK_DIR = "/content"
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab", "| results folder:", DRIVE_DIR)

GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Platform: Kaggle | results folder: /kaggle/working/agentguard


## Step 3: Install and start Ollama
This takes 1 to 2 minutes. You need to repeat it in every new Colab session.

In [5]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                    0.7%                                            39.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
import time, requests

log = open(f"{WORK_DIR}/ollama.log", "w")
server = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        if requests.get("http://localhost:11434").ok:
            print("Ollama server is running")
            break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print(f"Server did not start. Check {WORK_DIR}/ollama.log")

Ollama server is running


## Step 4: Pull the models
`qwen2.5:7b` is about 4.7 GB, so this takes a few minutes.

In [7]:
!ollama pull {MODEL}
!ollama pull {EMBED_MODEL}
!ollama list

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 2bada8a74506:   0% ▕                  ▏  11 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏  90 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   3% ▕                  ▏ 137 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   5% ▕                  ▏ 221 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   7% ▕█                 ▏ 304 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   7% ▕█                 ▏ 344 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   9% ▕█                 ▏ 441 MB/4.7 GB                  pulling m

In [8]:
!pip install -q ollama pandas

## Step 5: Sanity check
A plain chat call with no tools. The first call is slow because the model is loading into GPU memory.

In [9]:
import ollama

t0 = time.time()
r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": "Reply with one short sentence: are you working?"}])
print(r.message.content)
print(f"First call (includes model loading): {time.time() - t0:.1f}s")

Yes, I am working.
First call (includes model loading): 110.9s


## Step 6: Define three fake tools
These mirror tools AgentGuard will have later. They are only descriptions (schemas) for the model; nothing real happens.

| Tool | What it pretends to do |
|---|---|
| `get_weather(city)` | Look up the weather |
| `search_docs(query)` | Search Acme's company documents |
| `send_email(to, subject, body)` | Send an email |

In [10]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {"type": "object",
                       "properties": {"city": {"type": "string", "description": "City name"}},
                       "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "search_docs",
        "description": "Search Acme company documents (handbook, policies, product FAQ) and return matching passages.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string", "description": "What to search for"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "send_email",
        "description": "Send an email on behalf of the user.",
        "parameters": {"type": "object",
                       "properties": {"to": {"type": "string", "description": "Recipient email address"},
                                      "subject": {"type": "string"},
                                      "body": {"type": "string"}},
                       "required": ["to", "subject", "body"]}}},
]
SYSTEM = "You are Acme's internal assistant. Use a tool when it is needed to answer; otherwise answer directly."
print([t["function"]["name"] for t in TOOLS])

['get_weather', 'search_docs', 'send_email']


## Step 7: The 10-prompt tool-calling test
8 prompts need a specific tool with specific arguments. 2 prompts need **no** tool, because a good agent should not call tools when it doesn't have to.

A prompt passes only if the model picks the right tool **and** the key arguments contain the expected text. All calls use temperature 0 so the result is repeatable.

In [12]:
TESTS = [
    ("What's the weather in Chandigarh right now?",                         "get_weather", {"city": "chandigarh"}),
    ("Is it raining in Delhi today?",                                        "get_weather", {"city": "delhi"}),
    ("Mumbai mein aaj mausam kaisa hai?",                                    "get_weather", {"city": "mumbai"}),
    ("Find our leave policy in the company docs.",                           "search_docs", {"query": "leave"}),
    ("What does the product FAQ say about refunds?",                         "search_docs", {"query": "refund"}),
    ("Search the handbook for the work from home rules.",                    "search_docs", {"query": "home"}),
    ("Email meera@acme.local and tell her the report is ready.",             "send_email",  {"to": "meera@acme.local"}),
    ("Send arjun@acme.local an email with subject 'Meeting' saying the meeting moved to 4 pm.",
                                                                             "send_email",  {"to": "arjun@acme.local", "subject": "meeting"}),
    ("What is 15 multiplied by 4?",                                          None, {}),
    ("Say hello in Punjabi.",                                                None, {}),
]

def score(tool_calls, exp_tool, exp_args):
    if exp_tool is None:
        return (not tool_calls), "no tool expected"
    if not tool_calls:
        return False, "no tool called"
    call = tool_calls[0].function
    if call.name != exp_tool:
        return False, f"wrong tool: {call.name}"
    args = call.arguments or {}
    for k, v in exp_args.items():
        if v.lower() not in str(args.get(k, "")).lower():
            return False, f"bad argument {k}={args.get(k)!r}"
    return True, "ok"

In [13]:
import pandas as pd

rows = []
for i, (prompt, exp_tool, exp_args) in enumerate(TESTS, 1):
    t0 = time.time()
    r = ollama.chat(model=MODEL,
                    messages=[{"role": "system", "content": SYSTEM},
                              {"role": "user", "content": prompt}],
                    tools=TOOLS, options={"temperature": 0})
    secs = time.time() - t0
    calls = r.message.tool_calls or []
    ok, why = score(calls, exp_tool, exp_args)
    got = ", ".join(f"{c.function.name}({c.function.arguments})" for c in calls) or "(answered in text)"
    rows.append({"#": i, "prompt": prompt, "expected": exp_tool or "no tool", "got": got,
                 "pass": ok, "reason": why, "seconds": round(secs, 2)})
    print(f"{i:>2}. {'PASS' if ok else 'FAIL'}  {secs:5.1f}s  {got[:90]}")

df = pd.DataFrame(rows)
passed = int(df["pass"].sum())
print(f"\nScore: {passed}/10   |   average time per call: {df['seconds'].mean():.1f}s")
print("RESULT:", "GOOD - continue with this model" if passed >= 7 else "LOW - change MODEL in Step 1 and rerun from Step 4")
df

 1. PASS    0.9s  get_weather({'city': 'Chandigarh'})
 2. PASS    0.6s  get_weather({'city': 'Delhi'})
 3. PASS    0.6s  get_weather({'city': 'Mumbai'})
 4. PASS    0.6s  search_docs({'query': 'leave policy'})
 5. PASS    0.6s  search_docs({'query': 'refunds product FAQ'})
 6. PASS    0.6s  search_docs({'query': 'work from home rules'})
 7. PASS    1.9s  send_email({'to': 'meera@acme.local', 'subject': 'Your report is ready', 'body': 'Hello Me
 8. PASS    1.3s  send_email({'to': 'arjun@acme.local', 'subject': 'Meeting', 'body': 'The meeting has been 
 9. PASS    0.5s  (answered in text)
10. PASS    2.0s  (answered in text)

Score: 10/10   |   average time per call: 1.0s
RESULT: GOOD - continue with this model


,#,prompt,expected,got,pass,reason,seconds
0,1,What's the weather in Chandigarh right now?,get_weather,get_weather({'city': 'Chandigarh'}),True,ok,0.92
1,2,Is it raining in Delhi today?,get_weather,get_weather({'city': 'Delhi'}),True,ok,0.57
2,3,Mumbai mein aaj mausam kaisa hai?,get_weather,get_weather({'city': 'Mumbai'}),True,ok,0.59
3,4,Find our leave policy in the company docs.,search_docs,search_docs({'query': 'leave policy'}),True,ok,0.55
4,5,What does the product FAQ say about refunds?,search_docs,search_docs({'query': 'refunds product FAQ'}),True,ok,0.61
5,6,Search the handbook for the work from home rules.,search_docs,search_docs({'query': 'work from home rules'}),True,ok,0.62
6,7,Email meera@acme.local and tell her the report...,send_email,"send_email({'to': 'meera@acme.local', 'subject...",True,ok,1.94
7,8,Send arjun@acme.local an email with subject 'M...,send_email,"send_email({'to': 'arjun@acme.local', 'subject...",True,ok,1.26
8,9,What is 15 multiplied by 4?,no tool,(answered in text),True,no tool expected,0.45
9,10,Say hello in Punjabi.,no tool,(answered in text),True,no tool expected,2.03


## Step 8: A mini agent loop
This is the core of what `agent.py` will do later:
1. Send the user's question and the tools to the model
2. If the model asks for a tool, run it (here: a fake function) and send the result back
3. Repeat until the model answers in plain text (with a step limit, so it can't loop forever)

In [14]:
def get_weather(city):
    return f"{city}: 24°C, clear sky (fake data)"

def search_docs(query):
    return "Leave policy: employees get 18 days of paid leave per year (fake data)"

def send_email(to, subject, body):
    with open(f"{WORK_DIR}/outbox.txt", "a") as f:          # local file only, nothing is sent
        f.write(f"TO: {to} | SUBJECT: {subject} | BODY: {body}\n")
    return f"Email to {to} written to the local outbox"

FUNCS = {"get_weather": get_weather, "search_docs": search_docs, "send_email": send_email}

def run_agent(question, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(max_steps):
        r = ollama.chat(model=MODEL, messages=messages, tools=TOOLS, options={"temperature": 0})
        messages.append(r.message)
        if not r.message.tool_calls:
            return r.message.content
        for call in r.message.tool_calls:
            name, args = call.function.name, call.function.arguments
            result = FUNCS[name](**args) if name in FUNCS else f"Unknown tool {name}"
            print(f"  step {step+1}: {name}({args}) -> {result}")
            messages.append({"role": "tool", "content": str(result), "tool_name": name})
    return "(stopped: step limit reached)"

print(run_agent("How many days of paid leave do we get?"))

  step 1: search_docs({'query': 'paid leave policy'}) -> Leave policy: employees get 18 days of paid leave per year (fake data)
According to the company documents, employees get 18 days of paid leave per year.


## Step 9: Check the embedding model
RAG will use this model to turn document chunks into vectors. You should see a vector length (768 for `nomic-embed-text`).

In [15]:
e = ollama.embed(model=EMBED_MODEL, input=["Employees get 18 days of paid leave per year."])
print("Embedding length:", len(e.embeddings[0]))

Embedding length: 768


## Step 10: Save today's results

In [16]:
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
out = f"{DRIVE_DIR}/day1_toolcall_{MODEL.replace(':', '-')}_{stamp}.csv"
df.to_csv(out, index=False)
print("Saved:", out)
print(f"Write these down -> model: {MODEL} | score: {passed}/10 | avg seconds per call: {df['seconds'].mean():.1f}")

Saved: /kaggle/working/agentguard/day1_toolcall_qwen2.5-7b_20261110_1234.csv
Write these down -> model: qwen2.5:7b | score: 10/10 | avg seconds per call: 1.0


## Step 11: Create the GitHub repo (do this on your laptop)
1. Go to github.com → **New repository**. First search GitHub for "agentguard": the name is probably taken by other projects, so `agentguard-testbed` is a safe choice.
2. Set it to **Public**, tick **Add a README**, and create it.
3. Edit the README and add one line:
   `AgentGuard: a deliberately vulnerable LLM agent, an attack suite, and switchable defences that measure how much each defence lowers the attack success rate. Work in progress.`
4. Download this notebook (Colab: `File → Download → Download .ipynb`; Kaggle: `File → Download notebook`) and upload it to the repo as `notebooks/day1_setup.ipynb`.

## Day 1 checklist
- [ ] T4 GPU confirmed
- [ ] Ollama running, both models pulled
- [ ] Tool-calling score is 7/10 or higher (otherwise try the backup model)
- [ ] Mini agent loop answered using a tool
- [ ] Average seconds per call written down (needed to plan the evaluation runs)
- [ ] Results CSV saved (Drive on Colab, Output panel on Kaggle)
- [ ] Public GitHub repo created with README and this notebook

**Next (Day 2):** write the fake Acme data: handbook, leave policy, product FAQ, HR salary file with `CANARY-HR-7Q2X`, two mailboxes and five local web pages.